### Maximal Marginal Relevance
MMR (Maximal Marginal Relevance) is a powerful diversity-aware retrieval technique used in information retrieval and RAG pipelines to balance relevance and novelty when selecting documents.

In [4]:
from pathlib import Path
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableMap
from langchain_core.output_parsers import StrOutputParser


In [5]:
import os
from pathlib import Path
from dotenv import load_dotenv

dotenv_candidates = [
    Path(".env"),
    Path("../.env"),
    Path("09. Hybrid Search Strategies").parent / ".env",
]

dotenv_path = next((path.resolve() for path in dotenv_candidates if path.exists()), None)
if dotenv_path is None:
    raise FileNotFoundError("Could not find a .env file for loading API keys.")

load_dotenv(dotenv_path=dotenv_path)
groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError(f"GROQ_API_KEY was not found after loading {dotenv_path}")

print(f"Loaded environment from: {dotenv_path}")


Loaded environment from: /Users/shihab/Downloads/AI/Courses/01Portfolio/LangChainKrisnaik/playground/.env


In [6]:
# Step 1: Load and chunk the document
candidate_paths = [
    Path("09. Hybrid Search Strategies/7. langchain_rag_dataset.txt"),
    Path("7. langchain_rag_dataset.txt"),
]

file_path = next((path for path in candidate_paths if path.exists()), None)
if file_path is None:
    raise FileNotFoundError("Could not find 7. langchain_rag_dataset.txt from the current notebook working directory.")

loader = TextLoader(str(file_path))
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)
chunks


[Document(metadata={'source': '7. langchain_rag_dataset.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.'),
 Document(metadata={'source': '7. langchain_rag_dataset.txt'}, page_content='The framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.\nLangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.'),
 Document(metadata={'source': '7. langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
 Document(me

In [7]:
# Step 2: FAISS Vector Store with multilingual Hugging Face embeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
vectorstore = FAISS.from_documents(chunks, embedding_model)


In [8]:
### Step 3: Create MMR Retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 6, "lambda_mult": 0.5},
)


In [9]:
# Step 4: Prompt and LLM
prompt = PromptTemplate.from_template("""
Answer the question based on the context provided.

Context:
{context}

Question: {question}
""")
llm = init_chat_model("groq:llama-3.1-8b-instant")


In [10]:
# Step 5: RAG Pipeline
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    RunnableMap(
        {
            "context": lambda x: format_docs(retriever.invoke(x["question"])),
            "question": lambda x: x["question"],
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)


In [11]:
# Step 6: Query
query = {"question": "How does LangChain support agents and memory?"}
answer = rag_chain.invoke(query)
retrieved_docs = retriever.invoke(query["question"])

print("Answer:\n", answer)


Answer:
 LangChain supports agents and memory in the following ways:

1. **Agents**: LangChain agents can:
   - Use tools like calculators, search APIs, or custom functions based on the instructions they receive.
   - Interact with external APIs and databases, enhancing the capabilities of LLM-powered applications.
   - Decide which tool to call and in what order during a task, allowing LLMs to act as agents.

2. **Memory**: LangChain supports memory in two forms:
   - **Conversational Memory**: This is achieved through `ConversationBufferMemory` which helps models retain previous interactions, making multi-turn conversations more coherent.
   - **Summarization Memory**: This is achieved through `ConversationSummaryMemory` which enables summarization of previous conversations.


In [12]:
retrieved_docs


[Document(id='90307eac-fa3e-4ddf-b3af-fb8be4868c4f', metadata={'source': '7. langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
 Document(id='c306985f-6664-4c53-9685-98dbcc2e2a12', metadata={'source': '7. langchain_rag_dataset.txt'}, page_content='LangChain agents can interact with external APIs and databases, enhancing the capabilities of LLM-powered applications.\nRAG pipelines in LangChain involve document loading, splitting, embedding, retrieval, and LLM-based response generation.'),
 Document(id='4b784747-7379-496f-99ab-492b59b7c6cc', metadata={'source': '7. langchain_rag_dataset.txt'}, page_content='LangChain allows LLMs to act as agents that decide which tool to call and in what order during a task.\nLangChain supports conversational memory using Conve